# net-inspection-cv — GPU training on Colab

Runs the **exact same pipeline** as the repo, but on a Colab GPU so the heavy
experiments (multi-clip / bigger detectors / longer schedules) finish in minutes
instead of ~2 h on CPU. It clones the repo, pulls the real **SOLAQUA** footage
(public, CC BY-SA 4.0), rebuilds the *synthetic-damage-on-real-frames* dataset,
trains, and runs the held-out **adversarial / different-day** evaluation.

**How to use:** Runtime → Change runtime type → **GPU**, then Runtime → **Run all**.
Paste a GitHub token when the clone cell asks (the repo is private). Total time on
a T4: a few minutes of training after a one-time ~5–10 min SOLAQUA download.

**Honesty (unchanged):** all damage is synthetic (one generator). High numbers
here are a strong *proxy*, **not** validated real-damage performance.

**Best use of GPU credits:** the marginal same-day 3-clip run isn't worth much
(SOLAQUA has only 2 days). The notebook defaults to a **bigger detector**
(`yolov8s-seg`) for a real step up; cell 5 sketches the deferred SSL-pretraining.

## 1. GPU check + clone + install

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(out or 'No GPU — set Runtime > Change runtime type > GPU, then rerun.')

In [ ]:
# Private repo: paste a GitHub token (repo scope) when prompted.
import getpass, os, subprocess
TOKEN = getpass.getpass('GitHub token (blank if public): ').strip()
USER, REPO = 'Chrislysen', 'net-inspection-cv'
url = f'https://{TOKEN+"@" if TOKEN else ""}github.com/{USER}/{REPO}.git'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1',url], check=True)
%cd {REPO}
# Colab already ships GPU PyTorch — install everything EXCEPT torch so we never
# clobber the CUDA build, then add the package itself without re-resolving deps.
!pip -q install ultralytics rosbags opencv-python-headless scikit-image scikit-learn
!pip -q install -e . --no-deps
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 2. Pull SOLAQUA + build the composited dataset
Downloads real ROV bags (each ~1–2 GB) and composites labelled synthetic damage
onto disjoint real backgrounds. One **day** (2024-08-20) is held out for the
honest different-day test.

In [ ]:
CLIPS = {
  'solaqua_frames':  '2024-08-22_14-06-43_video.bag',
  'solaqua_bag2':    '2024-08-22_14-47-39_video.bag',
  'solaqua_bag3':    '2024-08-22_14-29-05_video.bag',
  'solaqua_diffday': '2024-08-20_15-18-27_video.bag',  # HELD OUT
}
import os, sys
sys.path.insert(0, 'src')
from netinspect import solaqua
name2id = {f.file_name: f.data_id for f in solaqua.list_files()}
for out, bag in CLIPS.items():
    if os.path.isdir(f'data/processed/{out}'):
        continue
    !python scripts/fetch_solaqua.py --data-id {name2id[bag]} --frames-out data/processed/{out} --every-n 6 --max-frames 200
print('frames:', {k: len(os.listdir(f'data/processed/{k}')) for k in CLIPS if os.path.isdir(f'data/processed/{k}')})

In [ ]:
# Train dataset from the day-1 clips; the held-out day + in/cross-clip sets are
# composited separately for the adversarial eval.
!python scripts/make_real_dataset.py --frames data/processed/solaqua_frames data/processed/solaqua_bag2 data/processed/solaqua_bag3 --out data/processed/multiclip_gpu --seg --damaged-fraction 0.85
!python scripts/make_real_dataset.py --frames data/processed/solaqua_diffday --out data/processed/diffday_composite --seg --damaged-fraction 0.85
!python scripts/make_real_dataset.py --frames data/processed/solaqua_frames --out data/processed/real_composite --seg
!python scripts/make_real_dataset.py --frames data/processed/solaqua_bag2 --out data/processed/bag2_composite --seg

## 3. Train on GPU
`yolov8s-seg.pt` (default) or `yolov8m-seg.pt` for more capacity. Minutes, not hours.

In [ ]:
import glob, os, shutil
MODEL  = 'yolov8s-seg.pt'   # bump to 'yolov8m-seg.pt' for more capacity
EPOCHS = 80
!python scripts/train_yolo.py --data data/processed/multiclip_gpu/dataset.yaml --task segment --model {MODEL} --epochs {EPOCHS} --imgsz 640 --batch 32 --device 0
best = sorted(glob.glob('runs/segment/*/weights/best.pt'), key=os.path.getmtime)[-1]
shutil.copy(best, 'models/yolo_damage_seg_gpu.pt')
print('saved', best)

## 4. Held-out adversarial / different-day evaluation

In [ ]:
!python scripts/adversarial_eval.py --yolo-weights models/yolo_damage_seg_gpu.pt --method yolo --out reports/results/adversarial_gpu
print(open('reports/results/adversarial_gpu/adversarial.md').read())

In [ ]:
# Download the trained weights, then send them back to wire into models/ and commit.
from google.colab import files
files.download('models/yolo_damage_seg_gpu.pt')

## 5. (High-value GPU use) SSL pretraining on SOLAQUA — the real deferred item
The repo only *probes* off-the-shelf DINOv2 features (`dino_backbone.py`). The
genuinely deferred experiment is **pretraining** a backbone on the unlabelled
SOLAQUA frames, then using it for anomaly detection / as a YOLO backbone:

```python
# pip install lightly  # self-supervised lib
# 1. Gather ALL undamaged SOLAQUA frames (no labels needed) into one folder.
# 2. Train DINO/SimCLR on them for a few hundred epochs (GPU, ~30-60 min).
# 3. Export the backbone; plug into patchcore.fit(...) via dino_backbone, or
#    initialise a YOLO backbone from it and fine-tune on the composited set.
# 4. Re-run scripts/compare_anomaly_backbones.py - does SOLAQUA-pretrained beat
#    off-the-shelf DINOv2 AND supervised ResNet on different-day AUROC?
```